[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HumbertoDiego/AjustamentoBasicoIME/blob/main/13_redes_geodesicas.ipynb)


# Ajustamento Básico - Redes geodésicas

**Maj Diego - 2° Semestre / 2026**

**Objetivos:**

1. Interpretar redes geodésicas como sistemas redundantes de observações.
2. Diferenciar injunções absolutas, relativas e mínimas.
3. Entender defeito de datum e redes livres.
4. Avaliar precisão e confiabilidade em redes ajustadas.

**Referências:**

- Ghilani, C. D. (2017). *Adjustment computations: Spatial data analysis* (6th ed.). Wiley. $\rightarrow$ **Cap. 15 e 16**
- Strang, G., & Borre, K. (1997). *Linear Algebra, Geodesy, and GPS*. Wellesley-Cambridge Press.


## O Problema

Uma rede geodésica combina observações redundantes para estimar coordenadas, altitudes ou outros parâmetros.

Além de ajustar observações, é preciso definir o datum: origem, orientação e escala do sistema de referência.


## 1. O que é uma rede geodésica?

Uma rede é um conjunto de pontos conectados por observações.

Exemplos:

- rede de nivelamento;
- rede GNSS;
- triangulação;
- trilateração;
- poligonal apoiada;
- rede hábrida com diferentes sensores.


### **1.1 Observações e parâmetros**

As observações podem ser distâncias, ângulos, desníveis, vetores GNSS, coordenadas observadas ou pseudo-observações.

Os parâmetros normalmente são coordenadas ou altitudes dos pontos.


## 2. Datum e injunções

O datum fixa os graus de liberdade que não são definidos apenas pelas observações relativas.

Em uma rede planimétrica, podem faltar translações, rotação e escala. Em uma rede altimétrica, pode faltar a origem vertical.


### **2.1 Tipos de injunção**

**Injunção absoluta:** fixa uma coordenada ou altitude em valor conhecido.

**Injunção relativa:** impõe relação entre pontos.

**Injunção mínima:** remove apenas o defeito de datum, sem superfixar a rede.

**Rede livre:** ajusta a geometria interna sem fixar rigidamente o datum externo.


## 3. Defeito de datum

Quando a matriz normal $N=A^TPA$ é singular, o sistema não possui solução única.

Em redes livres, essa singularidade pode ser esperada: a geometria relativa está definida, mas o datum não.

A solução exige injunções ou pseudoinversa.


## 4. Rede de nivelamento simples

Em uma rede de nivelamento, cada observação pode ser escrita como:

$$
H_j - H_i = \Delta h_{ij} + v_{ij}
$$

A matriz design recebe $-1$ na estação de origem e $+1$ na estação de destino.


**Exercício 01 (resolvido):** Ajuste uma pequena rede altimétrica apoiada em um ponto conhecido.


In [ ]:
import numpy as np

H_A = 100.000
obs = [
    ('A', 'B',  1.203, 0.004),
    ('B', 'C',  0.842, 0.005),
    ('A', 'C',  2.051, 0.006),
    ('C', 'D', -0.514, 0.004),
    ('B', 'D',  0.325, 0.007),
]
unknowns = ['B', 'C', 'D']

A = []
L = []
sig = []
for i, j, dh, s in obs:
    row = [0.0] * len(unknowns)
    const = dh
    if j in unknowns:
        row[unknowns.index(j)] += 1
    else:
        const -= H_A
    if i in unknowns:
        row[unknowns.index(i)] -= 1
    else:
        const += H_A
    A.append(row)
    L.append(const)
    sig.append(s)

A = np.array(A)
L = np.array(L).reshape(-1, 1)
P = np.diag(1 / np.array(sig)**2)

X = np.linalg.inv(A.T @ P @ A) @ A.T @ P @ L
V = A @ X - L

for nome, h in zip(unknowns, X.ravel()):
    print(f"H_{nome} = {h:.4f} m")
print("Resíduos (mm):", np.round(V.ravel()*1000, 2))


## 5. Precisão dos parâmetros

A matriz variância-covariância dos parâmetros ajustados é:

$$
\Sigma_X = \hat{\sigma}_0^2(A^TPA)^{-1}
$$

A diagonal fornece as variâncias das coordenadas ou altitudes ajustadas.


### **5.1 Elipses de erro**

Em redes planimétricas, cada ponto possui uma submatriz $2\times2$ de covariância.

A partir dela podem ser calculados o semieixo maior, o semieixo menor, a orientação da elipse e a região de confiança.


## 6. Confiabilidade da rede

Uma rede bem projetada deve permitir estimativas precisas, detecção de erros grosseiros, baixa influência de erros não detectados e geometria adequada das observações.

Precisão e confiabilidade não são sinônimos.


## 7. Redes livres e transformação de datum

Em uma rede livre, o ajustamento preserva a forma interna da rede.

Depois, a rede pode ser transformada para um datum externo usando pontos de controle.

Essa estratégia evita distorcer a geometria interna por controles inconsistentes.


**Exercício 02 (resolvido):** Observe a singularidade de uma rede altimétrica sem ponto fixo.


In [ ]:
unknowns = ['A', 'B', 'C', 'D']
A_livre = []
for i, j, dh, s in obs:
    row = [0.0] * len(unknowns)
    row[unknowns.index(j)] += 1
    row[unknowns.index(i)] -= 1
    A_livre.append(row)

A_livre = np.array(A_livre)
N = A_livre.T @ P @ A_livre

print("posto(A) =", np.linalg.matrix_rank(A_livre))
print("número de incógnitas =", A_livre.shape[1])
print("det(N) ≈", np.linalg.det(N))
print("A rede livre tem defeito de datum vertical igual a 1.")


## 8. Projeto de redes

Ao projetar uma rede, avalia-se antes da campanha:

- geometria das observações;
- redundância global e local;
- precisão esperada;
- pontos de controle;
- custo de campo;
- robustez contra perda de observações.


## Lista de exercícios complementares

**Exercício 03:** Ajuste uma rede de nivelamento com dois marcos conhecidos e compare com a solução usando apenas um marco conhecido.

**Exercício 04:** Monte uma rede planimétrica 2D com distâncias e estime coordenadas por MMQ não linear.

**Exercício 05:** Simule a remoção de uma observação em uma rede e avalie a mudança na precisão dos parâmetros.

**Exercício 06:** Pesquise um caso real de rede geodésica ou monitoramento estrutural e identifique observações, parâmetros, datum e controles de qualidade.
